# Analysis 模型差异 Review Briefing

目标：把 `model_effect_deltas` 中的 regime/window 模型差异转成稳定的中文 review briefing，让 notebook、dashboard 和 schedule 后续都能读取同一张分析表，而不是把解释逻辑放在 skill 或临时脚本中。

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / 'SPEC.md').exists() and (REPO_ROOT / 'src').exists()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import build_model_review_briefing, render_model_review_briefing_markdown

CARD = Path('research/analysis_model_review_briefing_v1')
SOURCE_DELTAS = Path('research/analysis_qlib_regime_v1/model_effect_deltas.csv')
CARD.mkdir(parents=True, exist_ok=True)
assert SOURCE_DELTAS.exists(), SOURCE_DELTAS


In [2]:
deltas = pd.read_csv(SOURCE_DELTAS)
briefing = build_model_review_briefing(deltas)
markdown = render_model_review_briefing_markdown(briefing, title='SOXL 模型差异 Review Briefing', max_rows=20)

briefing_path = CARD / 'model_review_briefing.csv'
markdown_path = CARD / 'model_review_briefing.md'
briefing.to_csv(briefing_path, index=False)
markdown_path.write_text(markdown, encoding='utf-8')

assert not briefing.empty
assert {'weakness', 'strength'}.issubset(set(briefing['review_signal']))
assert '## 中文结论' in markdown
assert '是否晋升' in markdown
briefing.head(10)


,schema_version,priority_rank,model,baseline_model,segment_type,segment,window,regime,review_signal,decision_hint,priority_score,relative_return_pct_delta,stability_score_delta,max_drawdown_pct_delta,turnover_delta,diagnosis,review_note
0,analysis.model_review_briefing.v1,1,alstm_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,alstm_mse 在 stress 弱于基准 logistic_clf：相对收益和稳定性同...
1,analysis.model_review_briefing.v1,2,autoformer_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,autoformer_mse 在 stress 弱于基准 logistic_clf：相对收益...
2,analysis.model_review_briefing.v1,3,darnn_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,darnn_mse 在 stress 弱于基准 logistic_clf：相对收益和稳定性同...
3,analysis.model_review_briefing.v1,4,dlinear_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,dlinear_mse 在 stress 弱于基准 logistic_clf：相对收益和稳定...
4,analysis.model_review_briefing.v1,5,fedformer_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,fedformer_mse 在 stress 弱于基准 logistic_clf：相对收益和...
5,analysis.model_review_briefing.v1,6,gat_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gat_mse 在 stress 弱于基准 logistic_clf：相对收益和稳定性同时落...
6,analysis.model_review_briefing.v1,7,gcn_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gcn_mse 在 stress 弱于基准 logistic_clf：相对收益和稳定性同时落...
7,analysis.model_review_briefing.v1,8,gru_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gru_mse 在 stress 弱于基准 logistic_clf：相对收益和稳定性同时落...
8,analysis.model_review_briefing.v1,9,hawkes_gru_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,hawkes_gru_mse 在 stress 弱于基准 logistic_clf：相对收益...
9,analysis.model_review_briefing.v1,10,hypergraph_mse,logistic_clf,regime,stress,,,weakness,block_or_rework,177.706487,-27.940035,-121.123791,1.176357,1.144492,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,hypergraph_mse 在 stress 弱于基准 logistic_clf：相对收益...


In [3]:
signal_counts = {str(k): int(v) for k, v in briefing['review_signal'].value_counts().sort_index().items()}
top_rows = briefing.head(5).to_dict(orient='records')
stress_rows = briefing[briefing['segment'].astype(str).str.contains('stress', na=False)]
summary = {
    'schema_version': 'research.analysis_model_review_briefing_v1',
    'source_deltas': str(SOURCE_DELTAS),
    'source_delta_rows': int(len(deltas)),
    'briefing_rows': int(len(briefing)),
    'signal_counts': signal_counts,
    'top_segments': [str(row['segment']) for row in top_rows],
    'top_models': [str(row['model']) for row in top_rows],
    'stress_review_rows': int(len(stress_rows)),
    'decision_hints': sorted(str(value) for value in briefing['decision_hint'].dropna().unique()),
    'artifacts': {
        'model_review_briefing': str(briefing_path),
        'model_review_markdown': str(markdown_path),
    },
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "schema_version": "research.analysis_model_review_briefing_v1",
  "source_deltas": "research/analysis_qlib_regime_v1/model_effect_deltas.csv",
  "source_delta_rows": 256,
  "briefing_rows": 175,
  "signal_counts": {
    "risk": 3,
    "strength": 25,
    "weakness": 147
  },
  "top_segments": [
    "stress",
    "stress",
    "stress",
    "stress",
    "stress"
  ],
  "top_models": [
    "alstm_mse",
    "autoformer_mse",
    "darnn_mse",
    "dlinear_mse",
    "fedformer_mse"
  ],
  "stress_review_rows": 27,
  "decision_hints": [
    "block_or_rework",
    "promote_candidate",
    "reduce_or_guardrail"
  ],
  "artifacts": {
    "model_review_briefing": "research/analysis_model_review_briefing_v1/model_review_briefing.csv",
    "model_review_markdown": "research/analysis_model_review_briefing_v1/model_review_briefing.md"
  }
}


## 中文结论

- 做了什么：基于 `research/analysis_qlib_regime_v1/model_effect_deltas.csv` 生成完整 `model_review_briefing.csv` 和中文 `model_review_briefing.md`，把模型差异解释沉到 `src.analysis.build_model_review_briefing`。
- 核心指标：从既有分段 delta 表中抽取 175 条 review 项，包含 `weakness`、`strength`、`risk` 等信号，并保留 `decision_hint`、相对收益差、稳定性差、回撤差和中文 review note。
- regime 差异：最高优先级集中在 stress 分段，说明多个序列/图/Transformer 代理模型在压力状态相对 logistic_clf 基准存在收益和稳定性弱点。
- 是否晋升：本实验只晋升 analysis review 框架能力；模型或策略不能仅凭 briefing 晋升到 production。
- 风险：briefing 的阈值仍是规则化启发，后续需要和 dashboard、schedule 通知以及更多真实回测窗口联动验证。
- 下一步：让 dashboard 读取 `model_review_briefing`，提供按 `review_signal` 和 `decision_hint` 的交互筛选。